In [41]:
from pathlib import Path
import build_lagged_features as blf   # works because the notebook is in the same folder as the .py file
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.formula.api import ols
from build_lagged_features import (
    DATE,
    DEFAULT_LAGS,
    TARGET,
    build_lagged_frame,
    read_table,
)
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler


def find_project_root(start=None):
    """Walk upward from the notebook's folder until a 'data/' folder is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "data").is_dir():
            return folder
    raise FileNotFoundError(f"No 'data' folder found above {start}")

DATA_DIR = find_project_root() / "data"                        # the folder
DATA_FILE = DATA_DIR / "chlorophyll_student_2015_2023.xlsx"    # the input file
print(DATA_DIR)
print(DATA_FILE)

df = blf.read_table(DATA_FILE)

C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data
C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\chlorophyll_student_2015_2023.xlsx


In [7]:
def make_lags(**overrides):
    """Copy of the default lags with changes.
    int N -> lags 0..N (1..N for the target); list -> used as is; None -> drop the variable."""
    spec = {k: list(v) for k, v in blf.DEFAULT_LAGS.items()}
    for col, val in overrides.items():
        if val is None:
            spec.pop(col, None)
        elif isinstance(val, int):
            spec[col] = list(range(1 if col == blf.TARGET else 0, val + 1))
        else:
            spec[col] = list(val)
    return spec

def build_and_save(raw, name, lag_spec=None, fmt="csv"):
    """Build the lagged frame and save it as <project>/data/<name>.<fmt>."""
    lagged = blf.build_lagged_frame(raw, lag_spec=lag_spec)
    path = DATA_DIR / f"{name}.{fmt}"
    if fmt == "xlsx":
        lagged.to_excel(path, sheet_name="lagged_data", index=False)
    else:
        lagged.to_csv(path, index=False)
    print(f"saved {lagged.shape} to {path}")
    return lagged

In [ ]:
'''
def analysis(lagged_data, model, scaling, split):
    # Separate features (X) and target (y)
    feature_cols = [c for c in lagged_data.columns if c not in [DATE, TARGET]]
    X = lagged_data[feature_cols]
    y = lagged_data[TARGET]
    # Chronological Train/Test Split (Train < 'split', Test >= 'split')
    train_mask = lagged_data[DATE] < split
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[~train_mask], y[~train_mask]
    if model=='LOS':
        model1 = LinearRegression(fit_intercept=True)
        model1.fit(X_train, y_train)
        y_pred = model1.predict(X_test)
        X = sm.add_constant(X_train)
        model2 = sm.OLS(y_train, X).fit()
        coefs = model2.summary2().tables[1]   # coef, std err, t, P>|t|, CI
        with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
            display(coefs)
        # only the significant ones, sorted by p-value
        display(coefs[coefs["P>|t|"] < 0.05].sort_values("P>|t|"))
    elif model=='Ridge':
        RidgeCV(alphas=alphas, cv=tscv)
    elif model == 'Lasso':
        LassoCV(alphas=alphas, cv=tscv, max_iter=10000, random_state=42)
    else:
        print(f"error: model not recognized")
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print({"RMSE": rmse, "MAE": mae, "R2": r2})
'''

# Linear Regression
def analysis_LOS(lagged_data, scaling, alpha, split):
    # Separate X and Y
    feature_cols = [c for c in lagged_data.columns if c not in [DATE, TARGET]]
    X = lagged_data[feature_cols]
    y = lagged_data[TARGET]
    # Chronological Train/Test Split (Train < 'split', Test >= 'split')
    train_mask = lagged_data[DATE] < split
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[~train_mask], y[~train_mask]
    model1 = LinearRegression(fit_intercept=True)
    model1.fit(X_train, y_train)
    y_pred = model1.predict(X_test)
    # Statistics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print({"RMSE": rmse, "MAE": mae, "R2": r2})
    # Summary
    X = sm.add_constant(X_train)
    model2 = sm.OLS(y_train, X).fit()
    coefs = model2.summary2().tables[1]   # coef, std err, t, P>|t|, CI
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
        display(coefs)
    # Summary with the significant ones, sorted by p-value
    display(coefs[coefs["P>|t|"] < alpha].sort_values("P>|t|"))
    
    

In [45]:
def analysis_LOS1(lagged_data, scaling, alpha, split):
    """scaling: None/False (no scaling), True or 'standard', 'minmax', 'robust'."""
    blf_DATE, blf_TARGET = blf.DATE, blf.TARGET
    feature_cols = [c for c in lagged_data.columns if c not in [blf_DATE, blf_TARGET]]
    X = lagged_data[feature_cols]
    y = lagged_data[blf_TARGET]

    # Chronological split
    train_mask = lagged_data[blf_DATE] < split
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[~train_mask], y[~train_mask]

    # Scaling (fit on train only, apply to test)
    if scaling:
        name = "standard" if scaling is True else scaling
        scaler = SCALERS[name]()
        X_train = pd.DataFrame(scaler.fit_transform(X_train),
                               columns=X_train.columns, index=X_train.index)
        X_test = pd.DataFrame(scaler.transform(X_test),
                              columns=X_test.columns, index=X_test.index)

    # sklearn model + test metrics
    model1 = LinearRegression(fit_intercept=True)
    model1.fit(X_train, y_train)
    y_pred = model1.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print({"RMSE": rmse, "MAE": mae, "R2": r2})

    # statsmodels OLS
    model2 = sm.OLS(y_train, sm.add_constant(X_train)).fit()
    summ = model2.summary()

    # First part of the summary (R-squared, F-statistic, AIC/BIC, ...)
    print(summ.tables[0])

    # All coefficients
    coefs = model2.summary2().tables[1]
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
        display(coefs)

    # Only the significant ones, sorted by p-value
    display(coefs[coefs["P>|t|"] < alpha].sort_values("P>|t|"))

    return model1, model2

In [ ]:
def analysis_Ridge(lagged_data, scaling, alpha, split):
    """scaling: None/False (no scaling), True or 'standard', 'minmax', 'robust'."""
    blf_DATE, blf_TARGET = blf.DATE, blf.TARGET
    feature_cols = [c for c in lagged_data.columns if c not in [blf_DATE, blf_TARGET]]
    X = lagged_data[feature_cols]
    y = lagged_data[blf_TARGET]

    # Chronological split
    train_mask = lagged_data[blf_DATE] < split
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[~train_mask], y[~train_mask]

    # Scaling (fit on train only, apply to test)
    if scaling:
        name = "standard" if scaling is True else scaling
        scaler = SCALERS[name]()
        X_train = pd.DataFrame(scaler.fit_transform(X_train),
                               columns=X_train.columns, index=X_train.index)
        X_test = pd.DataFrame(scaler.transform(X_test),
                              columns=X_test.columns, index=X_test.index)

    # sklearn model + test metrics
    model1 = RidgeCV(alphas=alphas, cv=tscv)
    model1.fit(X_train, y_train)
    y_pred = model1.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print({"RMSE": rmse, "MAE": mae, "R2": r2})

''' MUDAR PARA O RIDGE
    # statsmodels
    model2 = sm.OLS(y_train, sm.add_constant(X_train)).fit()
    summ = model2.summary()'''

    # First part of the summary (R-squared, F-statistic, AIC/BIC, ...)
    print(summ.tables[0])

    # All coefficients
    coefs = model2.summary2().tables[1]
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
        display(coefs)

    # Only the significant ones, sorted by p-value
    display(coefs[coefs["P>|t|"] < alpha].sort_values("P>|t|"))
    return model1, model2

In [ ]:
def analysis_Lasso(lagged_data, scaling, alpha, split):
    """scaling: None/False (no scaling), True or 'standard', 'minmax', 'robust'."""
    blf_DATE, blf_TARGET = blf.DATE, blf.TARGET
    feature_cols = [c for c in lagged_data.columns if c not in [blf_DATE, blf_TARGET]]
    X = lagged_data[feature_cols]
    y = lagged_data[blf_TARGET]

    # Chronological split
    train_mask = lagged_data[blf_DATE] < split
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[~train_mask], y[~train_mask]

    # Scaling (fit on train only, apply to test)
    if scaling:
        name = "standard" if scaling is True else scaling
        scaler = SCALERS[name]()
        X_train = pd.DataFrame(scaler.fit_transform(X_train),
                               columns=X_train.columns, index=X_train.index)
        X_test = pd.DataFrame(scaler.transform(X_test),
                              columns=X_test.columns, index=X_test.index)

    # sklearn model + test metrics
    model1 = RidgeCV(alphas=alphas, cv=tscv)
    model1.fit(X_train, y_train)
    y_pred = model1.predict(X_test)
    # Para X_test
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred)
    print(f"R2: {r2}\n"
        f"MAE: {mae}\n"
        f"MSE: {mse}\n"
        f"RMSE: {rmse}\n"
        f"MAPE: {mape}")

''' MUDAR PARA O RIDGE
    # statsmodels
    model2 = sm.OLS(y_train, sm.add_constant(X_train)).fit()
    summ = model2.summary()'''

    # First part of the summary (R-squared, F-statistic, AIC/BIC, ...)
    print(summ.tables[0])

    # All coefficients
    coefs = model2.summary2().tables[1]
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
        display(coefs)

    # Only the significant ones, sorted by p-value
    display(coefs[coefs["P>|t|"] < alpha].sort_values("P>|t|"))
    return

In [25]:
# Plots and Data Profiling

# Visualize data
df.head()
print(df.describe())
# Lista de variáveis numéricas do dataset
num_vars = [
    'chlorophyll_a_mg_m3', 'sst_c', 'par_umol_m2_s', 'nitrate_umol_l',
    'wind_speed_m_s', 'upwelling_index', 'mixed_layer_depth_m', 'salinity_psu',
    'current_speed_m_s', 'river_discharge_index', 'cloud_fraction',
    'surface_pressure_hpa', 'turbidity_ntu'
]

# Configuração de estilo do Seaborn
sns.set_theme(style="whitegrid")

'''
-----------------------------------------------------------------------------------------------------------------------
Boxplots
'''

# Boxplots Individuais
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(num_vars):
    sns.boxplot(
        y=df[col], 
        ax=axes[i], 
        color='skyblue', 
        flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.6)
    )
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('')
    axes[i].set_xlabel('')

# Remover os subplots não utilizados na grelha 4x4 (temos 13 variáveis)
for j in range(len(num_vars), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Boxplots per Variable', fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig("images/boxplots")
plt.close()


# Boxplot Conjunto Normalizado (Z-Score)
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[num_vars].dropna()), columns=num_vars)

plt.figure(figsize=(16, 8))
sns.boxplot(
    data=df_scaled, 
    orient='v', 
    palette='Set3',
    flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5)
)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.ylabel('Deviation', fontsize=12)
plt.title('Normalized Boxplots per Variable (Z-Score)', fontsize=15, fontweight='bold')
plt.axhline(y=3, color='r', linestyle='--', alpha=0.7, label='Limit Z = +3')
plt.axhline(y=-3, color='r', linestyle='--', alpha=0.7, label='Limit Z = -3')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig("images/boxplots_normalizados")
plt.close()


'''
-----------------------------------------------------------------------------------------------------------------------
Results and Codification
'''

print("--- Resumo da Estrutura dos Dados ---")
print(f"Número de registos: {len(df)}")
print(f"Intervalo de datas: {df['date'].min().date()} até {df['date'].max().date()}")
print(f"Valores em falta por coluna:\n{df.isnull().sum()}\n")

# Codificação Sazonal
day_of_year = df['date'].dt.dayofyear
df['s_year'] = np.sin(2 * np.pi * day_of_year / 365.25)
df['c_year'] = np.cos(2 * np.pi * day_of_year / 365.25)


'''
-----------------------------------------------------------------------------------------------------------------------
Analysis of Clorofila-a 
'''

# Clorofila-a - Analise da Variável (line plot)
plt.figure(figsize=(14, 5))
plt.plot(df['date'], df['chlorophyll_a_mg_m3'], color='forestgreen', linewidth=1)
plt.title('Temporal Trend Plot for Clorofila-a (2015–2023)')
plt.xlabel('Data')
plt.ylabel('Chlorophyll-a (mg/m³)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("images/Clorofila-a_Temporal_Trend_Plot")
plt.close()


# Clorofila-a - Histogram
plt.figure(figsize=(10, 5))
plt.hist(
    df['chlorophyll_a_mg_m3'].dropna(), 
    bins=70, 
    color='forestgreen', 
    edgecolor='black', 
    alpha=0.7
)
plt.title('Histogram of Clorofila-a')
plt.xlabel('Chlorophyll-a (mg/m³)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("images/Clorofila-a_histograma")
plt.close()


# Clorofila-a - Histogram with better fit distributions
# Prepare the data
data = df['chlorophyll_a_mg_m3'].dropna()
plt.figure(figsize=(10, 5))
# Plot histogram with density=True so the total area equals 1 (matching the PDF curves)
count, bins, ignored = plt.hist(
    data, 
    bins=50, 
    density=True, 
    color='forestgreen', 
    edgecolor='black', 
    alpha=0.5,
    label=' Observed Data (Histogram)'
)
# Generate a smooth range of x values for the fitted curves
x = np.linspace(data.min(), data.max(), 200)
# Fit and plot Lognormal distribution
shape_ln, loc_ln, scale_ln = stats.lognorm.fit(data, floc=0)
pdf_lognorm = stats.lognorm.pdf(x, shape_ln, loc_ln, scale_ln)
plt.plot(x, pdf_lognorm, 'r-', linewidth=2, label='Lognormal Fit')
# Fit and plot Gamma distribution
shape_g, loc_g, scale_g = stats.gamma.fit(data, floc=0)
pdf_gamma = stats.gamma.pdf(x, shape_g, loc_g, scale_g)
plt.plot(x, pdf_gamma, 'b--', linewidth=2, label='Gamma Fit')
plt.title('Histogram for Clorofila-a with Distributions')
plt.xlabel('Chlorophyll-a (mg/m³)')
plt.ylabel('Frequancy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("images/Clorofila-a_histograma_distributions")
plt.close()


'''
-----------------------------------------------------------------------------------------------------------------------
Histograms
'''

# Histograms of all variables
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(num_vars):
    sns.histplot(
        df[col].dropna(), 
        ax=axes[i], 
        color='forestgreen', 
        bins=30, 
        edgecolor='black',
        alpha=0.6
    )
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('')
    axes[i].set_xlabel('')
# Remove unused subplots in the 4x4 grid
for j in range(len(num_vars), len(axes)):
    fig.delaxes(axes[j])
plt.suptitle('Histograms per Variable', fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig("images/histograms")
plt.close()


# Histograms of all variables with best fit distributions
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(num_vars):
    data = df[col].dropna()
    # Plot density histogram
    axes[i].hist(data, bins=30, density=True, color='forestgreen', edgecolor='black', alpha=0.4)
    # Generate smooth curve points
    xmin, xmax = axes[i].get_xlim()
    x = np.linspace(xmin, xmax, 100)
    try:
        # Fit Lognormal if strictly positive, else Normal
        if (data > 0).all():
            shape, loc, scale = stats.lognorm.fit(data, floc=0)
            p = stats.lognorm.pdf(x, shape, loc, scale)
        else:
            mu, std = stats.norm.fit(data)
            p = stats.norm.pdf(x, mu, std)
        axes[i].plot(x, p, 'r-', linewidth=2)
    except Exception:
        pass  # Skip if fitting fails for a specific edge-case column
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('')
    axes[i].set_xlabel('')
for j in range(len(num_vars), len(axes)):
    fig.delaxes(axes[j])
plt.suptitle('Histograms with Best-Fit Distributions per Variable', fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig("images/histograms_distributions")
plt.close()


'''
-----------------------------------------------------------------------------------------------------------------------
Estacionariedade
'''

# Teste de Estacionariedade (ADF Test)
adf_result = adfuller(df['chlorophyll_a_mg_m3'].dropna(), result_object=True)
print(f"Estatística ADF: {adf_result[0]:.4f}")
print(f"p-value ADF: {adf_result[1]:.4e}")


'''
-----------------------------------------------------------------------------------------------------------------------
Correlation matrix and Scatter plot
'''

# Matriz de Correlacao entre Variaveis Exógenas e Alvo
exogenous_cols = [
    'sst_c', 'par_umol_m2_s', 'nitrate_umol_l', 'wind_speed_m_s',
    'upwelling_index', 'mixed_layer_depth_m', 'salinity_psu',
    'current_speed_m_s', 'river_discharge_index', 'cloud_fraction',
    'surface_pressure_hpa', 'turbidity_ntu'
]
corr_matrix = df[['chlorophyll_a_mg_m3'] + exogenous_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlação de Pearson')
plt.tight_layout()
plt.savefig("images/matrix_correlacao")
plt.close()


# Pairplot matrix for all numerical variables
pairplot_fig = sns.pairplot(
    df[num_vars].dropna(), 
    diag_kind='kde', 
    plot_kws={'alpha': 0.4, 's': 10, 'color': 'forestgreen'},
    diag_kws={'color': 'forestgreen'}
)
pairplot_fig.fig.suptitle('Scatter Plot Matrix', y=1.02, fontsize=16, fontweight='bold')
pairplot_fig.savefig("images/scatterplot.png", dpi=300)
plt.close()


'''
-----------------------------------------------------------------------------------------------------------------------
Correlation with Lags
'''


# Correlacao Cruzada com Lags das Exogenas (para selecionar lags no ARX)
lags = range(10, 15)
target = df['chlorophyll_a_mg_m3']

lag_corrs = pd.DataFrame(index=exogenous_cols, columns=[f'lag_{k}' for k in lags])

for col in exogenous_cols:
    for k in lags:
        lag_corrs.loc[col, f'lag_{k}'] = target.corr(df[col].shift(k))

plt.figure(figsize=(10, 6))
sns.heatmap(lag_corrs.astype(float), annot=True, fmt=".2f", cmap='viridis')
plt.title('Correlação Cruzada com Clorofila-a $y(t)$')
plt.xlabel('Atraso (Lags em dias)')
plt.tight_layout()
plt.savefig("images/correlacao_cruzada_lags")
plt.close()


                      date  chlorophyll_a_mg_m3        sst_c  par_umol_m2_s  \
count                 3287          3287.000000  3287.000000    3287.000000   
mean   2019-07-02 00:00:00             1.747175    15.302841     310.625146   
min    2015-01-01 00:00:00             0.276926     9.228039       8.000000   
25%    2017-04-01 12:00:00             1.312216    12.121438     149.255064   
50%    2019-07-02 00:00:00             1.624492    15.420263     311.322698   
75%    2021-09-30 12:00:00             2.074589    18.478341     470.651450   
max    2023-12-31 00:00:00            10.811834    21.275018     665.151866   
std                    NaN             0.633770     3.266127     169.246168   

       nitrate_umol_l  wind_speed_m_s  upwelling_index  mixed_layer_depth_m  \
count     3287.000000     3287.000000      3287.000000          3287.000000   
mean         4.402534        5.976806        -0.026088            23.929379   
min          0.811167        1.457805        -3.172

In [3]:
raw = blf.read_table(DATA_FILE)

# default lags
lagged = build_and_save(raw, "lagged_default")

# custom lags
my_lags = make_lags(
    chlorophyll_a_mg_m3=7,      # lags 1..7
    sst_c=[0, 1, 3, 7],         # specific lags
    turbidity_ntu=None,         # drop this variable
)
lagged_short = build_and_save(raw, "lagged_short", lag_spec=my_lags)

saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
saved (3273, 109) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_short.csv


In [46]:
# default lags
lagged = build_and_save(df, "lagged_default")
SCALERS = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}

analysis_LOS1(lagged, scaling="standard", alpha=0.05, split="2023-01-01")

saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
{'RMSE': np.float64(0.5471009284446116), 'MAE': 0.2243488116555656, 'R2': 0.5035694401864295}
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.758
Model:                             OLS   Adj. R-squared:                  0.747
Method:                  Least Squares   F-statistic:                     65.41
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         14:53:33   Log-Likelihood:                -638.13
No. Observations:                 2908   AIC:                             1544.
Df Residuals:                     2774   BIC:                             2345.
Df Model:                          133                                         
Covariance Type:             nonrobust                                         


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,1.738164,0.005722,303.794346,0.000000e+00,1.726946,1.749383
chlorophyll_a_mg_m3__lag_1,0.234752,0.011618,20.206216,8.195510e-85,0.211972,0.257533
chlorophyll_a_mg_m3__lag_2,0.018700,0.012448,1.502206,1.331578e-01,-0.005709,0.043109
chlorophyll_a_mg_m3__lag_3,0.006025,0.012459,0.483572,6.287282e-01,-0.018406,0.030456
chlorophyll_a_mg_m3__lag_4,-0.005259,0.012484,-0.421280,6.735835e-01,-0.029739,0.019220
chlorophyll_a_mg_m3__lag_5,-0.007033,0.012454,-0.564686,5.723332e-01,-0.031453,0.017388
chlorophyll_a_mg_m3__lag_6,0.017662,0.012446,1.419064,1.559928e-01,-0.006743,0.042066
chlorophyll_a_mg_m3__lag_7,0.028264,0.012337,2.290955,2.204068e-02,0.004073,0.052455
chlorophyll_a_mg_m3__lag_8,0.035533,0.012324,2.883204,3.966820e-03,0.011368,0.059698
chlorophyll_a_mg_m3__lag_9,0.004930,0.012322,0.400127,6.890941e-01,-0.019231,0.029091


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,1.738164,0.005722,303.794346,0.000000e+00,1.726946,1.749383
chlorophyll_a_mg_m3__lag_1,0.234752,0.011618,20.206216,8.195510e-85,0.211972,0.257533
wind_speed_m_s__lag_1,0.092790,0.016683,5.561769,2.925163e-08,0.060076,0.125503
nitrate_umol_l__lag_3,0.177816,0.034875,5.098664,3.650758e-07,0.109432,0.246199
par_umol_m2_s__lag_1,0.149075,0.031136,4.787904,1.773769e-06,0.088024,0.210127
upwelling_index__lag_2,0.113226,0.024672,4.589309,4.644842e-06,0.064849,0.161603
mixed_layer_depth_m__lag_1,-0.077820,0.023163,-3.359692,7.908457e-04,-0.123237,-0.032402
nitrate_umol_l__lag_4,0.105533,0.035064,3.009693,2.638664e-03,0.036778,0.174288
chlorophyll_a_mg_m3__lag_8,0.035533,0.012324,2.883204,3.966820e-03,0.011368,0.059698
chlorophyll_a_mg_m3__lag_7,0.028264,0.012337,2.290955,2.204068e-02,0.004073,0.052455


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x1603a505af0>)